# Bài tập 2 – Tiền xử lý và khai phá dữ liệu Online Retail

**Nguồn:** UCI Machine Learning Repository – Online Retail  
**Bài toán:** biến dữ liệu giao dịch thô thành thông tin có thể hỗ trợ quyết định kinh doanh.

Notebook trả lời 7 câu hỏi:

1. Dữ liệu đang có vấn đề gì?
2. Vấn đề nào cần giải quyết và vấn đề nào phải giữ lại để phân tích?
3. Làm sạch dữ liệu ra sao?
4. Đã thực hiện những phân tích nào?
5. Vì sao chọn RFM, K-Means và luật kết hợp?
6. Kết quả trước và sau tiền xử lý khác nhau thế nào?
7. Kết quả mong muốn và giới hạn của bài là gì?

> Dữ liệu gồm các giao dịch của một nhà bán lẻ trực tuyến tại Anh từ 01/12/2010 đến 09/12/2011.


## 1. Chuẩn bị môi trường

Các thư viện chính:

- `pandas`, `numpy`: đọc và xử lý dữ liệu;
- `matplotlib`: trực quan hóa;
- `scikit-learn`: chuẩn hóa, K-Means và đánh giá mô hình;
- `Counter`, `combinations`: tự xây dựng luật kết hợp theo cặp, không phụ thuộc `mlxtend`.


In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.style.use("seaborn-v0_8-whitegrid")

# Tự tìm file khi notebook nằm cạnh file Excel hoặc trong thư mục output/notebooks.
candidates = [
    Path("Online Retail.xlsx"),
    Path("upload/Online Retail.xlsx"),
    Path("../../upload/Online Retail.xlsx"),
    Path("../upload/Online Retail.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy Online Retail.xlsx. Hãy đặt file cạnh notebook "
        "hoặc sửa biến DATA_PATH."
    )
print("Đọc dữ liệu từ:", DATA_PATH.resolve())


ModuleNotFoundError: No module named 'matplotlib'

## 2. Đọc và hiểu dữ liệu thô

Mỗi dòng là một mặt hàng trong một hóa đơn. Một hóa đơn có thể có nhiều dòng.

| Thuộc tính | Ý nghĩa |
|---|---|
| `InvoiceNo` | Mã hóa đơn; bắt đầu bằng `C` là hóa đơn hủy |
| `StockCode` | Mã sản phẩm |
| `Description` | Tên sản phẩm |
| `Quantity` | Số lượng; âm thường biểu thị hoàn trả/hủy |
| `InvoiceDate` | Thời điểm giao dịch |
| `UnitPrice` | Đơn giá bảng Anh |
| `CustomerID` | Mã khách hàng |
| `Country` | Quốc gia của khách hàng |


In [5]:
df = pd.read_excel(DATA_PATH, sheet_name="Online Retail")
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
display(df.head())
display(df.dtypes.rename("Kiểu dữ liệu").to_frame())


NameError: name 'pd' is not defined

## 3. Dữ liệu có vấn đề gì?

Cần kiểm tra đồng thời:

- giá trị thiếu (`CustomerID`, `Description`);
- dòng trùng hoàn toàn;
- hóa đơn hủy (`InvoiceNo` bắt đầu bằng `C`);
- số lượng âm và giá không dương;
- dữ liệu không phải sản phẩm thật như phí bưu điện/phí ngân hàng;
- tháng 12/2011 chỉ có dữ liệu đến ngày 09/12 nên không được so sánh như một tháng đầy đủ.


In [ ]:
invoice_text = df["InvoiceNo"].astype(str)
cancelled = invoice_text.str.upper().str.startswith("C")

quality_before = pd.Series({
    "Số dòng": len(df),
    "Số hóa đơn": df["InvoiceNo"].nunique(),
    "Số sản phẩm": df["StockCode"].nunique(),
    "Số khách hàng có ID": df["CustomerID"].nunique(dropna=True),
    "Số quốc gia": df["Country"].nunique(),
    "Thiếu CustomerID": df["CustomerID"].isna().sum(),
    "Thiếu Description": df["Description"].isna().sum(),
    "Dòng trùng hoàn toàn": df.duplicated().sum(),
    "Dòng thuộc hóa đơn hủy": cancelled.sum(),
    "Quantity âm": (df["Quantity"] < 0).sum(),
    "UnitPrice <= 0": (df["UnitPrice"] <= 0).sum(),
}, name="Trước xử lý")

display(quality_before.to_frame())
print("Tỷ lệ thiếu CustomerID:", f"{df['CustomerID'].isna().mean():.2%}")
print("Khoảng thời gian:", df["InvoiceDate"].min(), "→", df["InvoiceDate"].max())


### Quyết định xử lý và lý do

1. **Xóa dòng trùng hoàn toàn** vì chúng làm doanh thu và tần suất mua bị đếm hai lần.
2. **Không điền giả `CustomerID`**. Trung bình/mode không có ý nghĩa với mã định danh và sẽ gộp nhầm nhiều người thành một khách hàng.
3. **Giữ dữ liệu thiếu `CustomerID` trong phân tích doanh thu**, nhưng loại khỏi RFM vì không thể biết giao dịch thuộc khách hàng nào.
4. **Tách giao dịch bán hợp lệ khỏi giao dịch hủy/hoàn trả**. Hủy trả là thông tin nghiệp vụ, không phải giá trị nhiễu để sửa thành số dương.
5. **Chỉ giữ `Quantity > 0`, `UnitPrice > 0`, có mô tả và không phải hóa đơn C** cho doanh thu bán dương, RFM và giỏ hàng.


In [ ]:
# 1) Loại dòng trùng hoàn toàn.
dedup = df.drop_duplicates().copy()

# 2) Chuẩn hóa kiểu mã và tạo biến doanh thu.
dedup["InvoiceNo"] = dedup["InvoiceNo"].astype(str).str.strip()
dedup["StockCode"] = dedup["StockCode"].astype(str).str.strip()
dedup["Revenue"] = dedup["Quantity"] * dedup["UnitPrice"]

# 3) Tập bán hàng hợp lệ để phân tích doanh thu dương.
valid_sale_mask = (
    ~dedup["InvoiceNo"].str.upper().str.startswith("C")
    & dedup["Description"].notna()
    & (dedup["Quantity"] > 0)
    & (dedup["UnitPrice"] > 0)
)
sales = dedup.loc[valid_sale_mask].copy()

# 4) Tập khách hàng: chỉ lấy các giao dịch có CustomerID thật.
customer_sales = sales.loc[sales["CustomerID"].notna()].copy()
customer_sales["CustomerID"] = customer_sales["CustomerID"].astype(int)

print(f"Sau khử trùng: {len(dedup):,} dòng")
print(f"Bán hàng hợp lệ: {len(sales):,} dòng")
print(f"Bán hàng có CustomerID: {len(customer_sales):,} dòng")


## 4. So sánh trước và sau tiền xử lý


In [ ]:
before_after = pd.DataFrame({
    "Chỉ tiêu": [
        "Số dòng", "Dòng trùng hoàn toàn", "Thiếu CustomerID",
        "Dòng hóa đơn hủy", "Quantity âm", "UnitPrice <= 0",
        "Số hóa đơn", "Số khách hàng có ID"
    ],
    "Trước xử lý": [
        len(df), df.duplicated().sum(), df["CustomerID"].isna().sum(),
        cancelled.sum(), (df["Quantity"] < 0).sum(),
        (df["UnitPrice"] <= 0).sum(), df["InvoiceNo"].nunique(),
        df["CustomerID"].nunique(dropna=True)
    ],
    "Sau xử lý (sales)": [
        len(sales), sales.duplicated().sum(), sales["CustomerID"].isna().sum(),
        sales["InvoiceNo"].str.startswith("C").sum(),
        (sales["Quantity"] < 0).sum(), (sales["UnitPrice"] <= 0).sum(),
        sales["InvoiceNo"].nunique(), sales["CustomerID"].nunique(dropna=True)
    ]
})
display(before_after)

revenue_summary = pd.Series({
    "Doanh thu dương hợp lệ": sales["Revenue"].sum(),
    "Doanh thu có CustomerID": customer_sales["Revenue"].sum(),
    "Tỷ lệ doanh thu có CustomerID": (
        customer_sales["Revenue"].sum() / sales["Revenue"].sum()
    ),
    "Doanh thu ròng gồm hoàn trả": dedup.loc[dedup["UnitPrice"] > 0, "Revenue"].sum(),
})
display(revenue_summary.to_frame("Giá trị"))


## 5. Khám phá doanh thu

Mục tiêu là thấy xu hướng theo thời gian, thị trường chính và sản phẩm phổ biến. Doanh thu được tính bằng:

$$Revenue = Quantity 	imes UnitPrice$$

Lưu ý: doanh thu ở đây là **doanh thu bán dương (gross positive sales)**; doanh thu ròng có hoàn trả được báo cáo riêng.


In [ ]:
monthly = (
    sales.assign(Month=sales["InvoiceDate"].dt.to_period("M").astype(str))
    .groupby("Month", as_index=False)
    .agg(Revenue=("Revenue", "sum"), Invoices=("InvoiceNo", "nunique"))
)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(monthly["Month"], monthly["Revenue"], marker="o", color="#3730a3")
ax.set(title="Doanh thu dương theo tháng", xlabel="Tháng", ylabel="Doanh thu (£)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
display(monthly)
print("Chú ý: 2011-12 chỉ có dữ liệu đến ngày 09/12.")


In [ ]:
country_revenue = (
    sales.groupby("Country", as_index=False)
    .agg(Revenue=("Revenue", "sum"), Invoices=("InvoiceNo", "nunique"))
    .sort_values("Revenue", ascending=False)
)
uk_share = country_revenue.loc[
    country_revenue["Country"].eq("United Kingdom"), "Revenue"
].iloc[0] / country_revenue["Revenue"].sum()
print("Tỷ trọng doanh thu UK:", f"{uk_share:.2%}")
display(country_revenue.head(10))

non_product_codes = {
    "POST", "DOT", "M", "BANK CHARGES", "AMAZONFEE", "CRUK",
    "D", "S", "PADS", "B", "C2"
}
product_sales = sales.loc[~sales["StockCode"].isin(non_product_codes)]
top_products = (
    product_sales.groupby(["StockCode", "Description"], as_index=False)
    .agg(InvoiceFrequency=("InvoiceNo", "nunique"),
         Quantity=("Quantity", "sum"), Revenue=("Revenue", "sum"))
    .sort_values("InvoiceFrequency", ascending=False)
)
display(top_products.head(10))


## 6. Phân khúc khách hàng bằng RFM + K-Means

- **Recency (R):** số ngày từ lần mua gần nhất đến ngày chốt dữ liệu; càng nhỏ càng tốt.
- **Frequency (F):** số hóa đơn khác nhau; càng lớn càng tốt.
- **Monetary (M):** tổng doanh thu khách hàng; càng lớn càng tốt.

Dữ liệu RFM thường lệch mạnh, vì vậy ta chặn ngoại lệ ở phân vị 99%, dùng `log1p`, rồi chuẩn hóa trước K-Means. Không chuẩn hóa sẽ khiến `Monetary` lấn át hai biến còn lại.


In [ ]:
snapshot = customer_sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
rfm = (
    customer_sales.groupby("CustomerID")
    .agg(LastPurchase=("InvoiceDate", "max"),
         Frequency=("InvoiceNo", "nunique"),
         Monetary=("Revenue", "sum"))
    .reset_index()
)
rfm["Recency"] = (snapshot - rfm["LastPurchase"].dt.normalize()).dt.days
rfm = rfm.loc[rfm["Monetary"] > 0].copy()

features = rfm[["Recency", "Frequency", "Monetary"]].copy()
for col in features:
    features[col] = features[col].clip(upper=features[col].quantile(0.99))
X = StandardScaler().fit_transform(np.log1p(features))

print("Ngày snapshot:", snapshot.date())
print("Số khách hàng dùng cho RFM:", f"{len(rfm):,}")
display(rfm[["Recency", "Frequency", "Monetary"]].describe())


In [ ]:
k_results = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = model.fit_predict(X)
    k_results.append({
        "k": k,
        "Silhouette": silhouette_score(X, labels),
        "Inertia": model.inertia_
    })
k_results = pd.DataFrame(k_results)
display(k_results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_results["k"], k_results["Silhouette"], marker="o", color="#3730a3")
ax.set(title="Đánh giá số cụm", xlabel="Số cụm k", ylabel="Silhouette")
plt.show()

print("k=2 có Silhouette cao nhất, nhưng chọn k=4 để tạo nhóm kinh doanh chi tiết hơn.")


In [ ]:
chosen_k = 4
model = KMeans(n_clusters=chosen_k, n_init=50, random_state=42)
rfm["Cluster"] = model.fit_predict(X)

profile = (
    rfm.groupby("Cluster")
    .agg(Customers=("CustomerID", "count"),
         RecencyMedian=("Recency", "median"),
         FrequencyMedian=("Frequency", "median"),
         MonetaryMedian=("Monetary", "median"),
         MonetaryTotal=("Monetary", "sum"))
    .reset_index()
)
profile["CustomerShare"] = profile["Customers"] / len(rfm)
profile["RevenueShare"] = profile["MonetaryTotal"] / rfm["Monetary"].sum()

# Đặt tên tự động dựa trên hồ sơ cụm để không phụ thuộc số nhãn ngẫu nhiên.
high_value = profile["MonetaryMedian"].idxmax()
dormant = profile["RecencyMedian"].idxmax()
remaining = [i for i in profile.index if i not in {high_value, dormant}]
core = max(remaining, key=lambda i: profile.loc[i, "FrequencyMedian"])
new_low = next(i for i in remaining if i != core)
profile["Segment"] = ""
profile.loc[high_value, "Segment"] = "Giá trị cao"
profile.loc[dormant, "Segment"] = "Ngủ quên"
profile.loc[core, "Segment"] = "Khách cốt lõi"
profile.loc[new_low, "Segment"] = "Mới/ít mua"

label_map = profile.set_index("Cluster")["Segment"].to_dict()
rfm["Segment"] = rfm["Cluster"].map(label_map)
display(profile.sort_values("RevenueShare", ascending=False))

# Kiểm tra độ ổn định qua nhiều random seed bằng ARI.
ari = []
for seed in range(10):
    alt = KMeans(n_clusters=4, n_init=1, random_state=seed).fit_predict(X)
    ari.append(adjusted_rand_score(rfm["Cluster"], alt))
print(f"ARI trung bình qua 10 seed: {np.mean(ari):.3f} ± {np.std(ari):.3f}")


**Hành động gợi ý theo phân khúc**

| Phân khúc | Hướng xử lý |
|---|---|
| Giá trị cao | VIP, ưu tiên chăm sóc và giữ chân |
| Khách cốt lõi | cross-sell, chương trình thành viên |
| Mới/ít mua | ưu đãi cho lần mua thứ hai |
| Ngủ quên | chiến dịch tái kích hoạt có giới hạn chi phí |


## 7. Luật kết hợp – sản phẩm nào thường xuất hiện cùng nhau?

Phân tích trên giỏ hàng tại UK vì UK chiếm phần lớn doanh thu và giúp giảm khác biệt thị trường.

- **Support:** tỷ lệ hóa đơn chứa cả A và B.
- **Confidence A→B:** trong các hóa đơn có A, tỷ lệ cũng có B.
- **Lift:** mức liên hệ so với việc B xuất hiện ngẫu nhiên; `lift > 1` là liên hệ dương.

Ngưỡng dùng trong bài: support ≥ 1%, confidence ≥ 25%, lift ≥ 1.2. Mã dịch vụ/điều chỉnh được loại khỏi giỏ hàng.


In [ ]:
basket_sales = sales.loc[
    sales["Country"].eq("United Kingdom")
    & ~sales["StockCode"].isin(non_product_codes)
].copy()

invoice_items = basket_sales.groupby("InvoiceNo")["StockCode"].agg(
    lambda s: sorted(set(s))
)
invoice_items = invoice_items.loc[invoice_items.map(len) >= 2]
n_baskets = len(invoice_items)

item_counts = Counter()
for items in invoice_items:
    item_counts.update(items)

frequent_items = {
    item for item, count in item_counts.items()
    if count / n_baskets >= 0.01
}

pair_counts = Counter()
for items in invoice_items:
    pair_counts.update(combinations(sorted(set(items) & frequent_items), 2))

description_map = (
    basket_sales.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .agg(lambda s: s.value_counts().index[0])
    .to_dict()
)

rules = []
for (a, b), pair_count in pair_counts.items():
    support = pair_count / n_baskets
    if support < 0.01:
        continue
    for ant, con in [(a, b), (b, a)]:
        confidence = pair_count / item_counts[ant]
        lift = confidence / (item_counts[con] / n_baskets)
        if confidence >= 0.25 and lift >= 1.2:
            rules.append({
                "Antecedent": description_map.get(ant, ant),
                "Consequent": description_map.get(con, con),
                "Support": support,
                "Confidence": confidence,
                "Lift": lift,
                "PairInvoices": pair_count,
            })

rules_df = pd.DataFrame(rules).sort_values(
    ["Lift", "Confidence", "Support"], ascending=False
)
print(f"Số giỏ hàng UK (>=2 sản phẩm): {n_baskets:,}")
print(f"Sản phẩm đạt support >=1%: {len(frequent_items):,}")
print(f"Số luật đạt ngưỡng: {len(rules_df):,}")
rules_display = rules_df.head(15).copy()
rules_display["Support"] = rules_display["Support"].map(lambda x: f"{x:.2%}")
rules_display["Confidence"] = rules_display["Confidence"].map(lambda x: f"{x:.2%}")
rules_display["Lift"] = rules_display["Lift"].map(lambda x: f"{x:.2f}")
display(rules_display)


## 8. Kết luận: đã làm gì và kết quả mong muốn

### Đã làm

1. Kiểm tra cấu trúc và chất lượng dữ liệu.
2. Khử trùng, nhận diện thiếu dữ liệu, hủy/hoàn trả và giá trị không hợp lệ.
3. Tạo hai tập dữ liệu đúng mục đích: `sales` và `customer_sales`.
4. Phân tích doanh thu theo thời gian, quốc gia và sản phẩm.
5. Phân khúc khách hàng bằng RFM + K-Means, kiểm tra Silhouette và độ ổn định ARI.
6. Tìm luật mua kèm bằng support, confidence và lift.

### Kết quả mong muốn

- Có dữ liệu sạch, có quy tắc xử lý rõ ràng và tái lập được.
- Biết khách hàng nào cần giữ chân, chăm sóc, tái kích hoạt.
- Biết sản phẩm nào nên gợi ý mua kèm hoặc bố trí gần nhau.
- Có số liệu doanh thu đáng tin cậy, đồng thời không che giấu giao dịch hoàn trả.

### Giới hạn cần ghi trong báo cáo

- Thiếu `CustomerID` khiến RFM chỉ bao phủ một phần doanh thu; không được suy rộng tuyệt đối.
- `k=2` có Silhouette cao nhất; `k=4` là lựa chọn ưu tiên khả năng hành động kinh doanh.
- Luật kết hợp thể hiện đồng xuất hiện, **không chứng minh quan hệ nhân quả**.
- Tháng 12/2011 là tháng chưa đầy đủ.
- Chưa có chi phí/lợi nhuận nên `Monetary` là doanh thu, không phải lợi nhuận.


In [ ]:
final_summary = pd.Series({
    "Dòng thô": len(df),
    "Dòng bán hợp lệ": len(sales),
    "Doanh thu dương hợp lệ (£)": sales["Revenue"].sum(),
    "Doanh thu ròng gồm hoàn trả (£)": dedup.loc[dedup["UnitPrice"] > 0, "Revenue"].sum(),
    "Khách hàng RFM": len(rfm),
    "Số phân khúc": chosen_k,
    "Giỏ hàng UK dùng cho luật kết hợp": n_baskets,
    "Số luật đạt ngưỡng": len(rules_df),
}, name="Kết quả")
display(final_summary.to_frame())
